# Chapter 02: Miscellaneous Math

Source span: printed pages 13-62; physical PDF pages 30-79.

This notebook turns the chapter's reference material into executable graphics math. The through-line is simple: a graphics program is safer when each mathematical idea has a representation that can be drawn, measured, and checked. We will use sets and mappings to talk about domains, quadratics to talk about robust intersections, vectors and frames to talk about orientation, implicit and parametric objects to talk about curves and surfaces, barycentric coordinates to talk about triangles, and probability to talk about Monte Carlo integration.

Chapter goal: build a compact computational vocabulary for later graphics chapters. By the end, you should be able to identify the set a quantity lives in, restrict a mapping until an inverse exists, choose a stable quadratic formula, repair an almost-frame into an orthonormal frame, read normals from gradients and cross products, reconstruct a point from interpolation weights, and audit a Monte Carlo estimator for normalization and variance.

The source chapter is broad on purpose. It is not one theorem chain; it is a toolbox. This notebook keeps that character, but every tool is attached to a figure or an invariant so the ideas are inspectable without opening the textbook.


In [ ]:
CHAPTER = 2
TITLE = "Miscellaneous Math"
TOPIC = f"chapter-{CHAPTER:02d}"
PRINTED_PAGES = "13-62"
PDF_PAGES = "30-79"
SOURCE_NOTE = "Fundamentals of Computer Graphics, Chapter 02, printed pp. 13-62 / physical PDF pp. 30-79"


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)


## Translation guide

| Chapter idea | Computational form used here | What can fail | Check used in the notebook |
| --- | --- | --- | --- |
| Set, mapping, domain, range, inverse | arrays plus functions with explicit domain restrictions | an inverse is requested where two inputs share one output | round-trip error after restricting the domain |
| Intervals and logarithms | interval endpoints, open/closed membership, inverse pairs | endpoint convention or base mismatch | interval intersection length and log/exp round trip |
| Quadratics and trigonometry | polynomial coefficients, roots, residuals, unit-circle identities | cancellation near a small root; degree/radian confusion | root product/residuals and exact SymPy identities |
| Vectors and coordinate frames | NumPy vectors, dot products, cross products, Gram matrices | non-unit vectors, left-handed frames, non-perpendicular axes | dot/cross invariants, Gram error, determinant sign |
| Curves and surfaces | implicit fields, gradients, parametric maps, surface tangents | normals point the wrong way or are not perpendicular | gradient/tangent and normal/tangent dot products |
| Linear interpolation and triangles | convex weights, barycentric coordinates, signed area | weights do not sum to one; outside points are accepted silently | reconstruction, area partition, orientation sign |
| Probability and density | PMFs, PDFs, CDFs, samples, estimator weights | probabilities do not normalize; estimator variance is hidden | normalization, expected value, Monte Carlo convergence |

## Visual storyboard

The planned visual sequence is deliberately concept-named rather than renderer-named:

1. `set-mapping-inverse-intervals.png`: shows why a mapping needs a domain restriction before an inverse is a function.
2. `quadratic-trigonometry-stability.png`: compares a naive quadratic formula with a cancellation-resistant one and anchors the trig identities used later.
3. `vectors-orthonormal-frame-repair.png`: connects dot product, cross product, area, handedness, and Gram-matrix frame checks.
4. `implicit-gradient-parametric-curve.png` and `parametric-surface-frame.html`: compare implicit gradients with parametric tangents and normals.
5. `linear-interpolation-barycentric-triangle.png`: makes interpolation weights visible on a segment and inside a triangle.
6. `probability-monte-carlo-importance.png`: tracks normalization, CDFs, estimator convergence, and importance sampling.
7. `triangle-monte-carlo-lab.png`: applies the chapter vocabulary to estimating an attribute average over a triangle.

Matplotlib is used for durable 2D and static 3D construction diagrams. Plotly is used for the parametric surface because a rotatable HTML surface better exposes tangent directions and normals than a fixed projection. SymPy is used for exact algebraic identities; NumPy is used for numerical geometry and Monte Carlo checks.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.graphics_math import barycentric_2d, normalize, signed_area2, stable_quadratic_roots, triangle_area
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#fbfcfe"})
rng = np.random.default_rng(202602)

figure_paths = []
html_paths = []
table_paths = []
json_paths = []
checks = {"source": SOURCE_NOTE, "chapter": CHAPTER, "title": TITLE}


def record_figure(path):
    figure_paths.append(Path(path))
    return path


def record_html(path):
    html_paths.append(Path(path))
    return path


def record_table(path):
    table_paths.append(Path(path))
    return path


def add_check(name, value):
    if isinstance(value, np.generic):
        value = value.item()
    checks[name] = value
    return value


## Sets, mappings, intervals, and inverses

A set tells us what objects are legal. A mapping tells us how an object from one set is sent to another set. The distinction matters in code because the same formula can behave like different functions when we change its domain.

The square function is the standard cautionary example. On the interval `[-2, 2]`, two different inputs often share one output, so a single-valued inverse is not available. On `[0, 2]`, the same formula becomes one-to-one and the square-root inverse works. The figure also treats intervals as computational objects: their endpoints and open/closed conventions determine which values a later routine may accept.


In [ ]:
x = np.linspace(-2.0, 2.0, 500)
query_y = 1.21
preimages = np.array([-math.sqrt(query_y), math.sqrt(query_y)])
restricted = np.linspace(0.0, 2.0, 300)
roundtrip = np.sqrt(restricted**2)
intersection = (0.20, 1.15)
log_points = np.linspace(0.1, 4.0, 80)
log_roundtrip_error = float(np.max(np.abs(np.exp(np.log(log_points)) - log_points)))

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.8))
axes[0].plot(x, x**2, color=PALETTE["blue"], linewidth=2)
axes[0].axhline(query_y, color=PALETTE["red"], linestyle="--", linewidth=1.2)
axes[0].scatter(preimages, preimages**2, color=PALETTE["red"], zorder=4)
for px in preimages:
    axes[0].plot([px, px], [0, query_y], color=PALETTE["gray"], linestyle=":", linewidth=1)
axes[0].text(-1.95, 3.55, "two inputs share one output", fontsize=9, color=PALETTE["ink"])
style_axis(axes[0], "formula on [-2, 2]: no single inverse", xlabel="input x", ylabel="output x^2")

axes[1].plot(restricted, restricted**2, color=PALETTE["teal"], label="f(x)=x^2 on [0,2]")
axes[1].plot(restricted**2, roundtrip, color=PALETTE["gold"], label="f^{-1}(y)=sqrt(y)")
axes[1].legend(fontsize=8, loc="upper left")
style_axis(axes[1], "restricted domain gives round trip", xlabel="domain or range coordinate", ylabel="mapped value")

for y0, lo, hi, color, label, left_closed, right_closed in [
    (0.65, -1.45, 1.15, PALETTE["blue"], "A=[-1.45,1.15)", True, False),
    (0.35, 0.20, 1.80, PALETTE["teal"], "B=(0.20,1.80]", False, True),
]:
    axes[2].hlines(y0, lo, hi, color=color, linewidth=8, alpha=0.72)
    axes[2].scatter([lo], [y0], s=70, facecolors=color if left_closed else "white", edgecolors=color, linewidth=1.5)
    axes[2].scatter([hi], [y0], s=70, facecolors=color if right_closed else "white", edgecolors=color, linewidth=1.5)
    axes[2].text(hi + 0.05, y0, label, va="center", fontsize=8)
axes[2].hlines(0.12, intersection[0], intersection[1], color=PALETTE["red"], linewidth=8, alpha=0.72)
axes[2].text(intersection[1] + 0.05, 0.12, "intersection used as safe domain", va="center", fontsize=8)
axes[2].set_ylim(-0.1, 0.9)
axes[2].set_yticks([])
style_axis(axes[2], "intervals as code contracts", xlabel="number line")

path = record_figure(save_matplotlib(fig, TOPIC, "set-mapping-inverse-intervals.png"))
close(fig)
add_check("mapping_ambiguous_preimage_count", int(len(preimages)))
add_check("restricted_square_inverse_roundtrip_max_error", float(np.max(np.abs(roundtrip - restricted))))
add_check("interval_intersection_length", float(intersection[1] - intersection[0]))
add_check("log_exp_roundtrip_max_error", log_roundtrip_error)

display_artifact(path, width=900)
checks["restricted_square_inverse_roundtrip_max_error"]


## Quadratics, angles, and stable roots

Ray intersections, visibility tests, and many geometric predicates eventually reduce to polynomial equations. The formula for a quadratic is exact algebraically, but the direct floating-point form can lose the small root when `b` and the square root of the discriminant nearly cancel.

The stable version computes one root using the sign of `b`, then gets the other from the product relation `r0*r1 = c/a`. The plot below compares the root product and residual behavior for a deliberately ill-conditioned quadratic. The same cell records exact trigonometric identities with SymPy because those identities are the link between angles, dot products, cross products, and solid-angle integrals later in the chapter.


In [ ]:
def naive_quadratic_roots(a, b, c):
    disc = b * b - 4.0 * a * c
    if disc < 0:
        return None
    root_disc = math.sqrt(disc)
    return tuple(sorted(((-b - root_disc) / (2.0 * a), (-b + root_disc) / (2.0 * a))))


def quadratic_residuals(a, b, c, roots):
    roots = np.asarray(roots, dtype=float)
    numerator = np.abs(a * roots**2 + b * roots + c)
    denominator = np.abs(a * roots**2) + np.abs(b * roots) + abs(c)
    return numerator / np.maximum(denominator, np.finfo(float).tiny)

a, b, c = 1.0, -1.0e8, 1.0
naive_roots = naive_quadratic_roots(a, b, c)
stable_roots = stable_quadratic_roots(a, b, c)
naive_product_error = abs(np.prod(naive_roots) - c / a)
stable_product_error = abs(np.prod(stable_roots) - c / a)
naive_resid = quadratic_residuals(a, b, c, naive_roots)
stable_resid = quadratic_residuals(a, b, c, stable_roots)

angle = np.linspace(0, 2*np.pi, 300)
unit = np.column_stack([np.cos(angle), np.sin(angle)])
theta = np.deg2rad(38)
pt = np.array([np.cos(theta), np.sin(theta)])
T = sp.symbols("T")
trig_identity = sp.simplify(sp.sin(T)**2 + sp.cos(T)**2 - 1)
double_identity = sp.simplify(sp.sin(2*T) - 2*sp.sin(T)*sp.cos(T))

fig, axes = plt.subplots(1, 3, figsize=(13.4, 3.9))
labels = ["small root", "large root"]
xloc = np.arange(2)
axes[0].bar(xloc - 0.17, naive_roots, width=0.34, color=PALETTE["red"], label="naive")
axes[0].bar(xloc + 0.17, stable_roots, width=0.34, color=PALETTE["teal"], label="stable")
axes[0].set_yscale("log")
axes[0].set_xticks(xloc, labels)
axes[0].legend(fontsize=8)
style_axis(axes[0], "roots span many scales", ylabel="root magnitude")

axes[1].bar(["naive", "stable"], [naive_product_error, stable_product_error + 1e-32], color=[PALETTE["red"], PALETTE["teal"]])
axes[1].set_yscale("log")
style_axis(axes[1], "root product should equal c/a", ylabel="absolute error")
axes[1].text(0.5, 0.8, "stable formula avoids cancellation", transform=axes[1].transAxes, ha="center", fontsize=9)

axes[2].plot(unit[:, 0], unit[:, 1], color=PALETTE["gray"], linewidth=1.5)
axes[2].plot([0, pt[0]], [0, pt[1]], color=PALETTE["blue"], linewidth=2)
axes[2].plot([pt[0], pt[0]], [0, pt[1]], color=PALETTE["teal"], linestyle="--", label="sin(theta)")
axes[2].plot([0, pt[0]], [0, 0], color=PALETTE["gold"], linewidth=3, label="cos(theta)")
axes[2].scatter([pt[0]], [pt[1]], color=PALETTE["red"], zorder=3)
axes[2].legend(fontsize=8, loc="lower left")
style_axis(axes[2], "unit circle stores trig ratios", equal=True, xlabel="x", ylabel="y")
axes[2].set_xlim(-1.12, 1.12)
axes[2].set_ylim(-1.12, 1.12)

path = record_figure(save_matplotlib(fig, TOPIC, "quadratic-trigonometry-stability.png"))
close(fig)
add_check("quadratic_naive_roots", [float(r) for r in naive_roots])
add_check("quadratic_stable_roots", [float(r) for r in stable_roots])
add_check("quadratic_naive_product_error", float(naive_product_error))
add_check("quadratic_stable_product_error", float(stable_product_error))
add_check("quadratic_stable_max_relative_residual", float(np.max(stable_resid)))
add_check("trig_pythagorean_identity", str(trig_identity))
add_check("trig_double_angle_identity", str(double_identity))

display_artifact(path, width=900)
{"naive_product_error": naive_product_error, "stable_product_error": stable_product_error, "trig_identity": trig_identity}


## Vectors, products, and orthonormal frames

A vector in graphics usually carries both magnitude and direction. The dot product measures the component of one vector in another direction. The cross product measures oriented area and returns a perpendicular direction in 3D. Together they make coordinate frames testable: an orthonormal frame has unit-length axes, zero off-diagonal dot products, and a positive determinant when it is right-handed.

Frame construction is a common place for silent bugs. If a normal vector is known, we can construct two perpendicular tangent directions. If two almost-perpendicular vectors are provided, we can repair them with a Gram-Schmidt-style process and then check the Gram matrix.


In [ ]:
a2 = np.array([1.18, 0.35])
b2 = np.array([0.32, 1.05])
area_cross_z = float(np.cross(np.append(a2, 0), np.append(b2, 0))[2])
dot_ab = float(np.dot(a2, b2))
angle_ab = float(np.arccos(dot_ab / (np.linalg.norm(a2) * np.linalg.norm(b2))))

w = normalize(np.array([0.34, 0.72, 0.60]))
helper = np.array([1.0, 0.0, 0.0]) if abs(w[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
u = normalize(np.cross(helper, w))
v = normalize(np.cross(w, u))
frame = np.column_stack([u, v, w])
gram = frame.T @ frame
basis_det = float(np.linalg.det(frame))

almost_a = normalize(np.array([0.98, 0.13, 0.07]))
almost_b = normalize(np.array([0.22, 0.94, 0.18]))
repaired_w = almost_a
repaired_u = normalize(almost_b - np.dot(almost_b, repaired_w) * repaired_w)
repaired_v = normalize(np.cross(repaired_w, repaired_u))
repaired = np.column_stack([repaired_u, repaired_v, repaired_w])
repaired_gram_error = float(np.max(np.abs(repaired.T @ repaired - np.eye(3))))

fig = plt.figure(figsize=(12.4, 4.4))
ax0 = fig.add_subplot(1, 2, 1)
poly = np.array([[0, 0], a2, a2 + b2, b2, [0, 0]], dtype=float)
ax0.fill(poly[:, 0], poly[:, 1], color=PALETTE["gold"], alpha=0.24, label="parallelogram area")
ax0.arrow(0, 0, a2[0], a2[1], head_width=0.05, length_includes_head=True, color=PALETTE["blue"], linewidth=2)
ax0.arrow(0, 0, b2[0], b2[1], head_width=0.05, length_includes_head=True, color=PALETTE["teal"], linewidth=2)
ax0.text(a2[0], a2[1], "a", color=PALETTE["blue"], fontsize=11)
ax0.text(b2[0], b2[1], "b", color=PALETTE["teal"], fontsize=11)
ax0.text(0.15, 1.45, f"a dot b = {dot_ab:.2f}\n|a x b| = {abs(area_cross_z):.2f}\nangle = {np.rad2deg(angle_ab):.1f} deg", fontsize=9)
style_axis(ax0, "dot product and cross-product area", equal=True, xlabel="x", ylabel="y")
ax0.set_xlim(-0.15, 1.75)
ax0.set_ylim(-0.15, 1.75)

ax1 = fig.add_subplot(1, 2, 2, projection="3d")
origin = np.zeros(3)
for vec, color, label in zip([u, v, w], [PALETTE["blue"], PALETTE["teal"], PALETTE["red"]], ["u tangent", "v tangent", "w normal"]):
    ax1.quiver(*origin, *vec, color=color, linewidth=2, arrow_length_ratio=0.12)
    ax1.text(*(vec * 1.08), label, color=color, fontsize=9)
ax1.set_title("constructed right-handed orthonormal frame")
ax1.set_xlim(-1, 1); ax1.set_ylim(-1, 1); ax1.set_zlim(-1, 1)
ax1.set_box_aspect((1, 1, 1))
ax1.view_init(elev=23, azim=38)
ax1.text2D(0.02, 0.02, f"max |G-I| = {np.max(np.abs(gram-np.eye(3))):.1e}\ndet = {basis_det:.2f}", transform=ax1.transAxes, fontsize=9)

path = record_figure(save_matplotlib(fig, TOPIC, "vectors-orthonormal-frame-repair.png"))
close(fig)
add_check("vector_dot_identity_error", float(abs(dot_ab - np.linalg.norm(a2) * np.linalg.norm(b2) * np.cos(angle_ab))))
add_check("cross_product_perpendicular_error", float(abs(np.dot(np.cross(np.append(a2, 0), np.append(b2, 0)), np.append(a2, 0)))))
add_check("constructed_frame_gram_error", float(np.max(np.abs(gram - np.eye(3)))))
add_check("constructed_frame_determinant", basis_det)
add_check("repaired_frame_gram_error", repaired_gram_error)

display_artifact(path, width=850)
{"frame_gram_error": checks["constructed_frame_gram_error"], "determinant": basis_det}


## Curves and surfaces: implicit tests versus parametric maps

The chapter gives two complementary ways to describe geometry. An implicit curve or surface is the zero set of a scalar field: points satisfy `f=0`, and the gradient points in the normal direction. A parametric curve or surface is a map from parameters into space: tangents come from differentiating the map, and a surface normal comes from the cross product of two tangent directions.

The static figure below puts these descriptions side by side for a 2D ellipse and a parametric curve. The Plotly artifact then gives a 3D parametric surface with normal segments so the tangent cross-product construction can be inspected by rotating the surface.


In [ ]:
gx, gy = np.meshgrid(np.linspace(-1.35, 1.35, 160), np.linspace(-1.55, 1.55, 160))
field = gx**2 + 0.55 * gy**2 - 1.0
phi = np.linspace(0, 2*np.pi, 24, endpoint=False)
contour_pts = np.column_stack([np.cos(phi), np.sin(phi) / math.sqrt(0.55)])
gradients = np.column_stack([2*contour_pts[:, 0], 1.10*contour_pts[:, 1]])
gradient_dirs = gradients / np.linalg.norm(gradients, axis=1, keepdims=True)
tangent_dirs = np.column_stack([-gradients[:, 1], gradients[:, 0]])
tangent_dirs = tangent_dirs / np.linalg.norm(tangent_dirs, axis=1, keepdims=True)
gradient_tangent_dots = np.sum(gradient_dirs * tangent_dirs, axis=1)
implicit_residual = contour_pts[:, 0]**2 + 0.55 * contour_pts[:, 1]**2 - 1.0

t = np.linspace(0, 2*np.pi, 500)
curve = np.column_stack([np.cos(t) + 0.24*np.cos(3*t), np.sin(t) - 0.18*np.sin(2*t)])
deriv = np.column_stack([-np.sin(t) - 0.72*np.sin(3*t), np.cos(t) - 0.36*np.cos(2*t)])
idx = np.linspace(20, len(t)-30, 12, dtype=int)
deriv_dirs = deriv[idx] / np.linalg.norm(deriv[idx], axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
axes[0].contour(gx, gy, field, levels=[0], colors=[PALETTE["blue"]], linewidths=2)
axes[0].contourf(gx, gy, field, levels=np.linspace(-1.0, 1.4, 16), cmap="RdBu_r", alpha=0.30)
axes[0].quiver(contour_pts[::2, 0], contour_pts[::2, 1], gradient_dirs[::2, 0], gradient_dirs[::2, 1], color=PALETTE["red"], scale=12, width=0.006)
axes[0].quiver(contour_pts[1::2, 0], contour_pts[1::2, 1], tangent_dirs[1::2, 0], tangent_dirs[1::2, 1], color=PALETTE["teal"], scale=12, width=0.006)
axes[0].text(-1.25, 1.26, "red: gradient normals\nteal: tangent directions", fontsize=9)
style_axis(axes[0], "implicit curve: gradient is normal", equal=True, xlabel="x", ylabel="y")

axes[1].plot(curve[:, 0], curve[:, 1], color=PALETTE["violet"], linewidth=2)
axes[1].quiver(curve[idx, 0], curve[idx, 1], deriv_dirs[:, 0], deriv_dirs[:, 1], color=PALETTE["gold"], scale=12, width=0.006)
axes[1].scatter(curve[idx, 0], curve[idx, 1], color=PALETTE["ink"], s=18)
style_axis(axes[1], "parametric curve: derivative is tangent", equal=True, xlabel="x(t)", ylabel="y(t)")

path = record_figure(save_matplotlib(fig, TOPIC, "implicit-gradient-parametric-curve.png"))
close(fig)
add_check("implicit_curve_max_residual", float(np.max(np.abs(implicit_residual))))
add_check("implicit_gradient_tangent_max_dot", float(np.max(np.abs(gradient_tangent_dots))))
add_check("parametric_curve_min_speed", float(np.min(np.linalg.norm(deriv, axis=1))))

display_artifact(path, width=850)
{"gradient_tangent_max_dot": checks["implicit_gradient_tangent_max_dot"], "curve_min_speed": checks["parametric_curve_min_speed"]}


In [ ]:
u_grid = np.linspace(0, 2*np.pi, 80)
v_grid = np.linspace(0, 2*np.pi, 36)
U, V = np.meshgrid(u_grid, v_grid)
R, r = 1.15, 0.34
X = (R + r*np.cos(V)) * np.cos(U)
Y = (R + r*np.cos(V)) * np.sin(U)
Z = r*np.sin(V)

sample_uv = [(0.35, 0.9), (1.7, 2.2), (3.7, 4.0), (5.2, 1.3)]
normal_lines = []
normal_tangent_dots = []
for uu, vv in sample_uv:
    p = np.array([(R + r*np.cos(vv))*np.cos(uu), (R + r*np.cos(vv))*np.sin(uu), r*np.sin(vv)])
    ru = np.array([-(R + r*np.cos(vv))*np.sin(uu), (R + r*np.cos(vv))*np.cos(uu), 0.0])
    rv = np.array([-r*np.sin(vv)*np.cos(uu), -r*np.sin(vv)*np.sin(uu), r*np.cos(vv)])
    n = normalize(np.cross(ru, rv))
    normal_tangent_dots.extend([float(np.dot(n, normalize(ru))), float(np.dot(n, normalize(rv)))])
    normal_lines.append((p, p + 0.32*n))

surface = go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", showscale=False, opacity=0.88)
traces = [surface]
for p, q in normal_lines:
    traces.append(go.Scatter3d(x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]], mode="lines+markers", line=dict(color="#c44e52", width=6), marker=dict(size=3), name="normal"))
fig3d = go.Figure(data=traces)
fig3d.update_layout(
    title="Parametric surface: normal = partial_u r x partial_v r",
    scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    margin=dict(l=0, r=0, t=42, b=0),
    showlegend=False,
)
html_path = record_html(save_plotly_html(fig3d, TOPIC, "parametric-surface-frame.html"))
add_check("parametric_surface_normal_tangent_max_dot", float(np.max(np.abs(normal_tangent_dots))))
add_check("parametric_surface_sample_normal_count", int(len(normal_lines)))

display_artifact(html_path, width="100%", height=520)
{"surface_normal_tangent_max_dot": checks["parametric_surface_normal_tangent_max_dot"]}


## Linear interpolation, triangles, and barycentric coordinates

Linear interpolation is a weighted average between two points: one weight goes down as the other goes up. Barycentric coordinates extend the same idea to triangles. A point has weights relative to the three vertices, and an affine quantity can be interpolated by applying the same weights to vertex values.

This is one of the most practical pieces of the chapter. Rasterization, texture lookup, vertex colors, normals, and many geometric tests become easier when triangle points are represented by weights that sum to one. The key warning is that the sum alone is not enough; negative weights reveal points outside the triangle.


In [ ]:
p0 = np.array([-0.20, 0.15])
p1 = np.array([1.25, 0.86])
ts = np.linspace(-0.25, 1.25, 13)
lerp_pts = (1-ts)[:, None] * p0 + ts[:, None] * p1

tri = np.array([[0.06, 0.10], [1.15, 0.25], [0.34, 1.03]])
weights = np.array([0.24, 0.31, 0.45])
p = weights @ tri
bary = barycentric_2d(p, *tri)
area_total = triangle_area(np.append(tri[0], 0), np.append(tri[1], 0), np.append(tri[2], 0))
sub_areas = np.array([
    triangle_area(np.append(p, 0), np.append(tri[1], 0), np.append(tri[2], 0)),
    triangle_area(np.append(tri[0], 0), np.append(p, 0), np.append(tri[2], 0)),
    triangle_area(np.append(tri[0], 0), np.append(tri[1], 0), np.append(p, 0)),
])

xs = np.linspace(-0.05, 1.18, 120)
ys = np.linspace(0.03, 1.08, 120)
points = np.array([[xx, yy] for yy in ys for xx in xs])
barys = np.array([barycentric_2d(q, *tri) for q in points])
inside = np.all(barys >= -1e-9, axis=1)
colors = np.clip(barys, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.8))
axes[0].plot([p0[0], p1[0]], [p0[1], p1[1]], color=PALETTE["blue"], linewidth=2)
axes[0].scatter(lerp_pts[:, 0], lerp_pts[:, 1], c=ts, cmap="plasma", s=46, edgecolor="white", linewidth=0.6)
axes[0].scatter([p0[0], p1[0]], [p0[1], p1[1]], color=PALETTE["ink"], zorder=4)
for label_t in [0, 0.5, 1.0]:
    q = (1-label_t)*p0 + label_t*p1
    axes[0].text(q[0], q[1] + 0.07, f"t={label_t:g}", fontsize=9, ha="center")
style_axis(axes[0], "lerp: p(t)=(1-t)p0+t p1", equal=True, xlabel="x", ylabel="y")

axes[1].scatter(points[inside, 0], points[inside, 1], c=colors[inside], s=6, alpha=0.82, linewidth=0)
loop = np.vstack([tri, tri[0]])
axes[1].plot(loop[:, 0], loop[:, 1], color=PALETTE["ink"], linewidth=1.6)
axes[1].scatter([p[0]], [p[1]], color=PALETTE["red"], s=62, zorder=4)
for label, vertex, wgt in zip(["a", "b", "c"], tri, bary):
    axes[1].text(vertex[0], vertex[1] + 0.045, f"{label}\n{wgt:.2f}", ha="center", fontsize=9)
axes[1].text(p[0] + 0.04, p[1], "point from weights", color=PALETTE["red"], fontsize=9)
style_axis(axes[1], "barycentric coordinates color the triangle", equal=True, xlabel="x", ylabel="y")

path = record_figure(save_matplotlib(fig, TOPIC, "linear-interpolation-barycentric-triangle.png"))
close(fig)
add_check("lerp_endpoint_error", float(np.linalg.norm(lerp_pts[ts == 0][0] - p0) + np.linalg.norm(lerp_pts[ts == 1][0] - p1)))
add_check("barycentric_sum", float(bary.sum()))
add_check("barycentric_reconstruction_error", float(np.linalg.norm(bary @ tri - p)))
add_check("triangle_area_partition_error", float(abs(sub_areas.sum() - area_total)))
add_check("triangle_signed_area_positive", bool(signed_area2(tri) > 0))

display_artifact(path, width=850)
{"barycentric": bary.round(6).tolist(), "area_partition_error": checks["triangle_area_partition_error"]}


## Probability, density, and Monte Carlo integration

Graphics uses probability when exact integration is too expensive. The crucial bookkeeping is normalization: a PMF must sum to one, a PDF must integrate to one, and an estimator must divide by the density that generated its samples.

The continuous example integrates `exp(-3x)` on `[0,1]`. Uniform sampling is unbiased but noisy because the integrand is concentrated near zero. Importance sampling draws more samples where the integrand is large and reweights each sample by `f(x)/p(x)`. When the PDF resembles the integrand, the estimator variance drops.


In [ ]:
outcomes = np.array([0, 1, 2, 3, 4], dtype=float)
pmf = np.array([0.08, 0.16, 0.31, 0.27, 0.18], dtype=float)
pmf = pmf / pmf.sum()
cdf = np.cumsum(pmf)
expected = float(np.sum(outcomes * pmf))
variance = float(np.sum((outcomes - expected)**2 * pmf))

x_pdf = np.linspace(0, 1, 400)
pdf = 2*x_pdf
cdf_cont = x_pdf**2
pdf_integral = float(np.trapz(pdf, x_pdf))

lam = 3.0
true_integral = float((1 - math.exp(-lam)) / lam)
Ns = np.array([8, 16, 32, 64, 128, 256, 512, 1024, 2048])
rows = []
for n in Ns:
    u = rng.random(n)
    uniform_est = float(np.mean(np.exp(-lam*u)))
    uniform_stderr = float(np.std(np.exp(-lam*u), ddof=1) / np.sqrt(n))
    z = rng.random(n)
    imp_x = -np.log(1 - z * (1 - math.exp(-lam))) / lam
    imp_pdf = lam * np.exp(-lam * imp_x) / (1 - math.exp(-lam))
    imp_weights = np.exp(-lam * imp_x) / imp_pdf
    rows.append({
        "samples": int(n),
        "uniform_estimate": uniform_est,
        "uniform_stderr": uniform_stderr,
        "importance_estimate": float(np.mean(imp_weights)),
        "importance_stderr": float(np.std(imp_weights, ddof=1) / np.sqrt(n)),
    })

csv_path = record_table(save_table_csv(rows, TOPIC, "monte-carlo-convergence.csv"))
uniform_errors = np.array([abs(row["uniform_estimate"] - true_integral) for row in rows])
importance_errors = np.array([abs(row["importance_estimate"] - true_integral) for row in rows])

fig, axes = plt.subplots(2, 2, figsize=(12.2, 7.4))
axes[0, 0].bar(outcomes, pmf, color=PALETTE["blue"], alpha=0.85, label="PMF")
axes[0, 0].step(outcomes, cdf, where="post", color=PALETTE["red"], label="CDF")
axes[0, 0].legend(fontsize=8)
style_axis(axes[0, 0], "discrete random variable", xlabel="outcome", ylabel="probability")

axes[0, 1].plot(x_pdf, pdf, color=PALETTE["teal"], label="PDF p(x)=2x")
axes[0, 1].plot(x_pdf, cdf_cont, color=PALETTE["gold"], label="CDF x^2")
axes[0, 1].legend(fontsize=8)
style_axis(axes[0, 1], "continuous density on [0,1]", xlabel="x", ylabel="density / probability")

axes[1, 0].plot(Ns, [row["uniform_estimate"] for row in rows], "o-", color=PALETTE["blue"], label="uniform")
axes[1, 0].plot(Ns, [row["importance_estimate"] for row in rows], "o-", color=PALETTE["red"], label="importance")
axes[1, 0].axhline(true_integral, color=PALETTE["ink"], linestyle="--", linewidth=1, label="exact")
axes[1, 0].set_xscale("log", base=2)
axes[1, 0].legend(fontsize=8)
style_axis(axes[1, 0], "Monte Carlo estimates", xlabel="sample count", ylabel="integral estimate")

axes[1, 1].plot(Ns, uniform_errors, "o-", color=PALETTE["blue"], label="uniform error")
axes[1, 1].plot(Ns, importance_errors + 1e-16, "o-", color=PALETTE["red"], label="importance error")
axes[1, 1].set_xscale("log", base=2)
axes[1, 1].set_yscale("log")
axes[1, 1].legend(fontsize=8)
style_axis(axes[1, 1], "error after density reweighting", xlabel="sample count", ylabel="absolute error")

path = record_figure(save_matplotlib(fig, TOPIC, "probability-monte-carlo-importance.png"))
close(fig)
add_check("discrete_pmf_sum", float(pmf.sum()))
add_check("discrete_expected_value", expected)
add_check("discrete_variance", variance)
add_check("continuous_pdf_integral_trapezoid", pdf_integral)
add_check("monte_carlo_true_integral_exp_minus_3x", true_integral)
add_check("monte_carlo_uniform_final_error", float(uniform_errors[-1]))
add_check("monte_carlo_importance_final_error", float(importance_errors[-1]))
add_check("importance_sampling_variance_ratio", float(np.var(importance_errors) / max(np.var(uniform_errors), 1e-30)))

display_artifact(path, width=860)
display_artifact(csv_path)
{"true_integral": true_integral, "final_uniform_error": uniform_errors[-1], "final_importance_error": importance_errors[-1]}


## Applied lab: average an attribute over a triangle

This lab combines the chapter's pieces into a miniature graphics task. Suppose a triangle has a scalar attribute stored at its three vertices, such as brightness, height, opacity, or a single color channel. Barycentric coordinates interpolate the attribute at any point. Probability supplies random points over the triangle. Monte Carlo estimates the triangle average.

For an affine attribute, the exact average over a triangle is the average of the three vertex values. The estimator below samples uniformly over the triangle, evaluates the barycentric interpolation, and checks that the estimate approaches the exact value. This is the same mental model used later for sampling pixels, surfaces, lights, and texture domains.


In [ ]:
lab_tri = np.array([[0.08, 0.12], [1.20, 0.18], [0.46, 1.08]])
vertex_values = np.array([0.20, 0.92, 0.48])
exact_triangle_average = float(vertex_values.mean())
N_lab = 1800
u_rand = rng.random(N_lab)
v_rand = rng.random(N_lab)
sqrt_u = np.sqrt(u_rand)
lab_bary = np.column_stack([1 - sqrt_u, sqrt_u * (1 - v_rand), sqrt_u * v_rand])
lab_points = lab_bary @ lab_tri
lab_values = lab_bary @ vertex_values
running_est = np.cumsum(lab_values) / np.arange(1, N_lab + 1)
lab_stderr = float(np.std(lab_values, ddof=1) / np.sqrt(N_lab))
lab_error = float(abs(running_est[-1] - exact_triangle_average))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
scatter = axes[0].scatter(lab_points[:, 0], lab_points[:, 1], c=lab_values, s=8, cmap="viridis", alpha=0.62, linewidth=0)
loop = np.vstack([lab_tri, lab_tri[0]])
axes[0].plot(loop[:, 0], loop[:, 1], color=PALETTE["ink"], linewidth=1.4)
for label, vertex, value in zip(["A", "B", "C"], lab_tri, vertex_values):
    axes[0].text(vertex[0], vertex[1] + 0.04, f"{label}: {value:.2f}", ha="center", fontsize=9)
fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04, label="interpolated attribute")
style_axis(axes[0], "uniform barycentric samples over one triangle", equal=True, xlabel="x", ylabel="y")

axes[1].plot(np.arange(1, N_lab + 1), running_est, color=PALETTE["blue"], linewidth=1.5)
axes[1].axhline(exact_triangle_average, color=PALETTE["red"], linestyle="--", linewidth=1.2, label="exact vertex average")
axes[1].fill_between(np.arange(1, N_lab + 1), exact_triangle_average - 2*lab_stderr, exact_triangle_average + 2*lab_stderr, color=PALETTE["gold"], alpha=0.20, label="final +/- 2 stderr")
axes[1].legend(fontsize=8)
style_axis(axes[1], "Monte Carlo triangle average", xlabel="samples", ylabel="estimated mean")

path = record_figure(save_matplotlib(fig, TOPIC, "triangle-monte-carlo-lab.png"))
close(fig)
add_check("triangle_lab_exact_attribute_average", exact_triangle_average)
add_check("triangle_lab_estimated_attribute_average", float(running_est[-1]))
add_check("triangle_lab_standard_error", lab_stderr)
add_check("triangle_lab_absolute_error", lab_error)
add_check("triangle_lab_barycentric_sum_max_error", float(np.max(np.abs(lab_bary.sum(axis=1) - 1))))
add_check("triangle_lab_inside_min_weight", float(np.min(lab_bary)))

display_artifact(path, width=850)
{"estimate": running_est[-1], "exact": exact_triangle_average, "standard_error": lab_stderr}


## Sanity checks

The final cell makes the notebook audit itself. It checks every generated artifact, tests that PNGs are nonblank, and asserts core invariants from each section. The numeric tolerances are intentionally tight for exact constructions and looser where Monte Carlo randomness is part of the experiment.


In [ ]:
invariant_path = save_json(checks, TOPIC, "chapter-02-invariant-summary.json")
json_paths.append(invariant_path)

all_artifacts = [*figure_paths, *html_paths, *table_paths, *json_paths]
artifact_records = assert_artifacts(all_artifacts)
image_records = [assert_nonblank_image(path) for path in figure_paths]

assert checks["restricted_square_inverse_roundtrip_max_error"] < 1e-12
assert checks["interval_intersection_length"] > 0
assert checks["log_exp_roundtrip_max_error"] < 1e-12
assert checks["quadratic_stable_product_error"] < checks["quadratic_naive_product_error"]
assert checks["trig_pythagorean_identity"] == "0"
assert checks["trig_double_angle_identity"] == "0"
assert checks["constructed_frame_gram_error"] < 1e-12
assert checks["constructed_frame_determinant"] > 0.99
assert checks["repaired_frame_gram_error"] < 1e-12
assert checks["implicit_curve_max_residual"] < 1e-12
assert checks["implicit_gradient_tangent_max_dot"] < 1e-12
assert checks["parametric_surface_normal_tangent_max_dot"] < 1e-12
assert abs(checks["barycentric_sum"] - 1.0) < 1e-12
assert checks["barycentric_reconstruction_error"] < 1e-12
assert checks["triangle_area_partition_error"] < 1e-12
assert checks["triangle_signed_area_positive"] is True
assert abs(checks["discrete_pmf_sum"] - 1.0) < 1e-12
assert abs(checks["continuous_pdf_integral_trapezoid"] - 1.0) < 5e-6
assert checks["monte_carlo_uniform_final_error"] < 0.035
assert checks["monte_carlo_importance_final_error"] < 1e-12
assert checks["importance_sampling_variance_ratio"] < 1e-8
assert checks["triangle_lab_barycentric_sum_max_error"] < 1e-12
assert checks["triangle_lab_inside_min_weight"] >= 0
assert checks["triangle_lab_absolute_error"] < 3.0 * checks["triangle_lab_standard_error"]

final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "source_note": SOURCE_NOTE,
    "figure_artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in figure_paths],
    "html_artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in html_paths],
    "table_artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in table_paths],
    "check_artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in json_paths],
    "nonblank_image_count": len(image_records),
    "artifact_count": len(artifact_records),
    "key_checks": {
        "frame_gram_error": checks["constructed_frame_gram_error"],
        "barycentric_reconstruction_error": checks["barycentric_reconstruction_error"],
        "pdf_integral": checks["continuous_pdf_integral_trapezoid"],
        "importance_final_error": checks["monte_carlo_importance_final_error"],
        "triangle_lab_absolute_error": checks["triangle_lab_absolute_error"],
    },
}
final_path = save_json(final_report, TOPIC, "final-sanity.json")
assert_artifacts([final_path])
display_artifact(final_path)
final_report


## Takeaways

The chapter's topics are miscellaneous only at the surface. In graphics code, they form one chain of responsibility:

- A domain restriction can turn an ambiguous formula into a usable inverse.
- A stable quadratic solve can be the difference between a missed hit and a correct intersection.
- Dot products, cross products, and Gram matrices make coordinate frames auditable.
- Implicit geometry gives membership tests and gradients; parametric geometry gives traversal, tangents, and surface normals.
- Linear and barycentric interpolation turn geometric locations into weights for colors, normals, depths, texture coordinates, and attributes.
- Probability is not decoration around rendering; it is the bookkeeping that keeps sampled estimates unbiased.

The artifacts and checks saved by this notebook are intentionally small. They are meant to be re-run after edits, because the habits here carry into ray tracing, rasterization, shading, sampling, and texture mapping later in the course.
